In [1]:
# !pip install streamlit-tensorboard

In [4]:
import tensorflow as tf
print(tf.__version__)
import ipykernel
import pandas
import numpy
import sklearn
import tensorboard
import matplotlib
import streamlit

print("ipykernel:", ipykernel.__version__)
print("pandas:", pandas.__version__)
print("numpy:", numpy.__version__)
print("scikit-learn:", sklearn.__version__)
print("tensorboard:", tensorboard.__version__)
print("matplotlib:", matplotlib.__version__)
print("streamlit:", streamlit.__version__)

2.20.0
ipykernel: 6.30.1
pandas: 2.3.3
numpy: 1.26.4
scikit-learn: 1.7.2
tensorboard: 2.20.0
matplotlib: 3.10.6
streamlit: 1.50.0


In [3]:
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import pandas as pd
import numpy as np


In [4]:
import tensorflow as tf

In [5]:
## Load the data
df=pd.read_csv("Churn_Modelling.csv")

In [6]:
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
## Prreprocessing the data
df.drop(columns=["RowNumber","CustomerId","Surname"],axis=1,inplace=True)

In [8]:
## Encode categorial data variables
label_gender=LabelEncoder()
df["Gender"]=label_gender.fit_transform(df["Gender"])
df.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [9]:
df['Geography'].value_counts()

Geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64

In [10]:
# One hot encoding
from sklearn.preprocessing import OneHotEncoder

encoded_geo=OneHotEncoder()
encoded=encoded_geo.fit_transform(df[["Geography"]])
encoded_geo_df=pd.DataFrame(encoded.toarray(),columns=encoded_geo.get_feature_names_out(["Geography"]))
encoded_geo_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [11]:
df=pd.concat([df,encoded_geo_df],axis=1)
df.drop(columns=["Geography"],axis=1,inplace=True)
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [12]:
import pickle

## Save the scaled data

with open("01_encoded_gender.pkl","wb") as f:
    pickle.dump(label_gender,f)
with open('02_encoded_geo.pkl','wb') as f:
    pickle.dump(encoded_geo,f)

In [13]:
## Divide the dataset into features and target variable
X=df.drop(columns=["Exited"],axis=1)
y=df["Exited"]

In [14]:
## Split the data into training and testing sets
x_train,x_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

## scale the data
scaler=StandardScaler()
x_train_scaled=scaler.fit_transform(x_train)
x_test_scaled=scaler.transform(x_test)

In [15]:
with open("03_scaler.pkl","wb") as f:
    pickle.dump(scaler,f)

## ANN Implementation

In [16]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

In [17]:
x_train.columns

Index(['CreditScore', 'Gender', 'Age', 'Tenure', 'Balance', 'NumOfProducts',
       'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_France',
       'Geography_Germany', 'Geography_Spain'],
      dtype='object')

In [18]:
## Build ANN Model
model =Sequential([
    Dense(64,activation='relu',input_shape=(x_train.shape[1],)), # First Hidden Layer
    # Dense(32, activation='relu'), # HL2
    Dense(16, activation='relu'),
    # Dense(8, activation='relu'),
    Dense(4, activation='relu'),
    Dense(2, activation='relu'),
    Dense(1, activation='sigmoid') # Output layer
])

/home/sheshpalsingh/anaconda3/envs/myenv/lib/python3.10/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-04-05 11:51:48.375721: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [19]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │            68 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 2)              │            10 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │             3 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,953 (7.63 KB)

 Trainable params: 1,953 (7.63 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
##Optimizer
opt=tf.keras.optimizers.Adam(learning_rate=0.01)
loss=tf.keras.losses.BinaryCrossentropy()

In [21]:
## Compile the model
model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])


In [22]:
## Setup the tensorboard
log_dir='logs/fit'+datetime.datetime.now().strftime("%Y%m%d")
# log_dir='logs/fit'+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

In [23]:
import tensorflow

tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [24]:
## Setup Early Stopping
early_stopping_callback=EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [25]:
## Training the model
history=model.fit(
    x_train,y_train, validation_data=(x_test,y_test),epochs=100,
    callbacks=[tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - accuracy: 0.7945 - loss: 6.6879 - val_accuracy: 0.8035 - val_loss: 0.4980
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7945 - loss: 0.5082 - val_accuracy: 0.8035 - val_loss: 0.4959
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7945 - loss: 0.5080 - val_accuracy: 0.8035 - val_loss: 0.4956
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7945 - loss: 0.5081 - val_accuracy: 0.8035 - val_loss: 0.4956
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7945 - loss: 0.5081 - val_accuracy: 0.8035 - val_loss: 0.4956
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7945 - loss: 0.5082 - val_accuracy: 0.8035 - val_loss: 0.4961
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7945 - loss: 0.5081 - val_accuracy: 0.8035 - val_loss: 0.4957
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.7945 - loss: 0.5081 - val_accu

In [26]:
model.save('04_model.h5')

In [27]:
## Load Tensorboard
%load_ext tensorboard

In [29]:
%tensorboard --logdir logs/fit20260405/train/